# 09 — Scenario Comparison Dashboard

**Purpose:** Side-by-side comparison of all completed E4ST scenarios on key
policy metrics: zonal LMPs, generation mix (%), CO₂ emissions, and carbon
intensity.  Policy levers are a $50/tCO₂ carbon tax and a 50 % clean-energy
standard (the 80 % CES target was infeasible — see `network_metadata.json`).

This notebook is strictly visualisation — all values are read directly from
Julia parquet outputs.  It is designed to be re-run after any new scenario
is added to `data/processed/e4st_results/`.

**Inputs:**
- `data/processed/e4st_results/{scenario}/lmp.parquet`
- `data/processed/e4st_results/{scenario}/dispatch.parquet`
- `data/processed/network_metadata.json`

**Outputs:**
- `data/processed/scenario_lmp_compare.png`
- `data/processed/scenario_mix_share.png`
- `data/processed/scenario_emissions.png`
- `data/processed/scenario_summary.csv`

In [ ]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RESULTS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'e4st_results'
META_PATH   = PROJECT_ROOT / 'data' / 'processed' / 'network_metadata.json'

with open(META_PATH) as f:
    meta = json.load(f)

SCENARIOS = [
    s['name']
    for s in meta['scenarios_completed']
    if s['status'] == 'OPTIMAL'
]

SCENARIO_LABELS = {
    'baseline':       'Baseline',
    'carbon_tax_50':  'Carbon Tax $50/tCO₂',
    'ces_achievable': 'CES 50% (max feasible)',
}

FUEL_COLORS = {
    'coal':       '#2d2d2d',
    'ng':         '#4169e1',
    'nuclear':    '#d62728',
    'wind':       '#2ca02c',
    'solar':      '#ff7f0e',
    'hydro':      '#17becf',
    'biomass':    '#8c564b',
    'oil':        '#9467bd',
    'geothermal': '#e377c2',
    'storage':    '#bcbd22',
    'other':      '#7f7f7f',
}
FUEL_ORDER = [
    'ng', 'coal', 'nuclear', 'hydro', 'wind', 'solar',
    'biomass', 'geothermal', 'oil', 'storage', 'other'
]

# Note any infeasible scenarios for display
INFEASIBLE = [
    s['name']
    for s in meta['scenarios_completed']
    if s['status'] == 'INFEASIBLE'
]

print('Completed (OPTIMAL):', SCENARIOS)
print('Infeasible:         ', INFEASIBLE)

In [ ]:
# ── Load all results ───────────────────────────────────────────────────────────
lmp_all      = {}
dispatch_all = {}

for sc in SCENARIOS:
    lmp_all[sc]      = pd.read_parquet(RESULTS_DIR / sc / 'lmp.parquet')
    dispatch_all[sc] = pd.read_parquet(RESULTS_DIR / sc / 'dispatch.parquet')

# Combined dispatch with scenario label
dispatch = pd.concat(
    [df.assign(scenario=sc) for sc, df in dispatch_all.items()],
    ignore_index=True
)
dispatch['dispatch_twh'] = dispatch['dispatch_mwh'] / 1e9
dispatch['co2_mtons']    = dispatch['co2_emitted_tons'] / 1e9

fuels_present = [f for f in FUEL_ORDER if f in dispatch['genfuel'].unique()]
print('Loaded scenarios:', list(lmp_all.keys()))
print('Fuel types:      ', fuels_present)

In [ ]:
# ── Summary metrics table ─────────────────────────────────────────────────────
agg = (
    dispatch
    .groupby('scenario')[['dispatch_twh', 'co2_mtons']]
    .sum()
)

rows = []
for sc in SCENARIOS:
    ldf = lmp_all[sc]
    total_twh  = agg.loc[sc, 'dispatch_twh']
    total_co2  = agg.loc[sc, 'co2_mtons'] * 1000   # → Mt CO2
    carbon_int = (total_co2 * 1e6) / (total_twh * 1e9)  # tCO2/MWh
    rows.append({
        'Scenario':                    SCENARIO_LABELS.get(sc, sc),
        'Mean LMP ($/MWh)':            ldf['lmp_mwh'].mean(),
        'Min LMP ($/MWh)':             ldf['lmp_mwh'].min(),
        'Max LMP ($/MWh)':             ldf['lmp_mwh'].max(),
        'Total Generation (TWh)':      total_twh,
        'Total CO₂ (Mt)':              total_co2,
        'Carbon Intensity (tCO₂/MWh)': carbon_int,
    })

summary_df = pd.DataFrame(rows).set_index('Scenario')

# Save CSV
csv_path = PROJECT_ROOT / 'data' / 'processed' / 'scenario_summary.csv'
summary_df.to_csv(csv_path)
print(f'Saved → {csv_path}')

summary_df.style.format({
    'Mean LMP ($/MWh)':            '{:.2f}',
    'Min LMP ($/MWh)':             '{:.2f}',
    'Max LMP ($/MWh)':             '{:.2f}',
    'Total Generation (TWh)':      '{:.1f}',
    'Total CO₂ (Mt)':              '{:.0f}',
    'Carbon Intensity (tCO₂/MWh)': '{:.4f}',
})

In [ ]:
# ── Figure 1: LMP by BA — all scenarios side-by-side ─────────────────────────
# Sort BAs by baseline LMP (descending) for a consistent axis order
base_lmp = lmp_all['baseline'].set_index('ba')['lmp_mwh'].sort_values(ascending=False)
ba_order  = base_lmp.index.tolist()

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(ba_order))
width = 0.28
offsets = np.linspace(-width, width, len(SCENARIOS))

colors = ['#1f77b4', '#d62728', '#2ca02c']

for i, sc in enumerate(SCENARIOS):
    ldf = lmp_all[sc].set_index('ba')['lmp_mwh']
    vals = [ldf.get(ba, np.nan) for ba in ba_order]
    ax.bar(x + offsets[i], vals, width=width * 0.95,
           color=colors[i], label=SCENARIO_LABELS.get(sc, sc),
           alpha=0.85, edgecolor='white', linewidth=0.3)

ax.set_xticks(x)
ax.set_xticklabels(ba_order, rotation=90, fontsize=6.5)
ax.set_ylabel('LMP ($/MWh)')
ax.set_title('Zonal LMPs by BA — All Scenarios (sorted by Baseline LMP)')
ax.legend(frameon=False)
ax.set_xlim(-0.6, len(ba_order) - 0.4)

fig.tight_layout()
out = PROJECT_ROOT / 'data' / 'processed' / 'scenario_lmp_compare.png'
fig.savefig(out, bbox_inches='tight')
print(f'Saved → {out}')
plt.show()

In [ ]:
# ── Figure 2: 100% stacked generation mix — scenario comparison ───────────────
gen_agg = (
    dispatch
    .groupby(['scenario', 'genfuel'])['dispatch_twh']
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=fuels_present, fill_value=0)
)

# Normalise to %
gen_share = gen_agg.div(gen_agg.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(SCENARIOS))
bottoms = np.zeros(len(SCENARIOS))

for fuel in fuels_present:
    vals = gen_share.reindex(SCENARIOS)[fuel].values
    bars = ax.bar(x, vals, bottom=bottoms,
                  color=FUEL_COLORS.get(fuel, '#aaa'),
                  label=fuel, edgecolor='white', linewidth=0.4)
    # Label segments > 5%
    for xi, (bot, val) in enumerate(zip(bottoms, vals)):
        if val >= 5:
            ax.text(xi, bot + val / 2, f'{val:.0f}%',
                    ha='center', va='center', fontsize=7.5,
                    color='white' if fuel in ('coal', 'ng', 'nuclear') else 'black')
    bottoms += vals

ax.set_ylim(0, 100)
ax.set_xticks(x)
ax.set_xticklabels([SCENARIO_LABELS.get(s, s) for s in SCENARIOS])
ax.set_ylabel('Share of Generation (%)')
ax.set_title('Generation Mix — Scenario Comparison (100% stacked)')

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1],
          bbox_to_anchor=(1.01, 1), loc='upper left',
          frameon=False, fontsize=9)

fig.tight_layout()
out = PROJECT_ROOT / 'data' / 'processed' / 'scenario_mix_share.png'
fig.savefig(out, bbox_inches='tight')
print(f'Saved → {out}')
plt.show()

In [ ]:
# ── Figure 3: CO₂ emissions + carbon intensity — dual-axis ───────────────────
fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax2 = ax1.twinx()

x = np.arange(len(SCENARIOS))
bar_w = 0.45

# CO2 bars on primary axis
co2_vals = [
    dispatch[dispatch['scenario'] == sc]['co2_mtons'].sum() * 1000
    for sc in SCENARIOS
]
ci_vals = [
    summary_df.loc[SCENARIO_LABELS.get(sc, sc), 'Carbon Intensity (tCO₂/MWh)']
    for sc in SCENARIOS
]

bars = ax1.bar(x, co2_vals, width=bar_w, color='#555555',
               alpha=0.75, label='Total CO₂ (Mt)', zorder=3)

# Carbon intensity line on secondary axis
ax2.plot(x, ci_vals, 'o-', color='#d62728', linewidth=2, markersize=7,
         label='Carbon Intensity (tCO₂/MWh)', zorder=4)
ax2.set_ylabel('Carbon Intensity (tCO₂/MWh)', color='#d62728')
ax2.tick_params(axis='y', colors='#d62728')
ax2.spines['right'].set_visible(True)
ax2.spines['right'].set_color('#d62728')

ax1.set_xticks(x)
ax1.set_xticklabels([SCENARIO_LABELS.get(s, s) for s in SCENARIOS])
ax1.set_ylabel('Total CO₂ Emissions (Mt)')
ax1.set_title('CO₂ Emissions & Carbon Intensity — Scenario Comparison')

# Combined legend
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, frameon=False, fontsize=9, loc='upper right')

# Annotate bars with values
for xi, val in enumerate(co2_vals):
    ax1.text(xi, val * 1.01, f'{val:.0f} Mt',
             ha='center', va='bottom', fontsize=8)

fig.tight_layout()
out = PROJECT_ROOT / 'data' / 'processed' / 'scenario_emissions.png'
fig.savefig(out, bbox_inches='tight')
print(f'Saved → {out}')
plt.show()

In [ ]:
# ── Policy interpretation notes ───────────────────────────────────────────────
lmp_passthrough = meta['carbon_tax_lmp_passthrough_per_mwh']
ces_max         = meta['ces_max_feasible_fraction']

print('── Policy Notes ─────────────────────────────────────────────────────────')
print(f'Carbon tax $50/tCO₂ → LMP passthrough: ${lmp_passthrough:.2f}/MWh')
print(f'  (tax raises marginal cost of gas; shadow price passes through to all zones)')
print()
print(f'CES target: 80 % clean energy — status: INFEASIBLE')
print(f'Max feasible CES fraction: {ces_max:.0%}')
print(f'  (current fleet renewable/nuclear capacity cannot reach 80% in a single period;')
print(f'   the solver was re-run at {ces_max:.0%} — "ces_achievable" scenario)')
print()
print(f'ces_achievable LMPs are identical to baseline because the {ces_max:.0%} CES')
print(f'constraint is non-binding given the current generation fleet.')